# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

In [ ]:
# List all Record Sets (referenced by `@id`)
record_sets = metadata.record_set
print(f"Number of record sets found: {len(record_sets)}\n")

for idx, rs in enumerate(record_sets):
    print(f"RecordSet {idx}: @id = {rs.id}, name = '{rs.name}'")
    print("  Fields:")
    for field in rs.field:
        print(f"    - @id: {field.id}\n      name: {field.name}\n      dataType: {field.data_type}\n      column: {[c.id for c in field.column]}\n      description: {field.description}\n")
    print("-"*40)


## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in metadata.record_set]
dataframes = {}

for rs in metadata.record_set:
    print(f"Loading records for RecordSet: {rs.name} (id={rs.id})")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    print(df.head(2))
    print('-'*30)

# Pick the first record set for detailed demonstration
active_record_set_id = record_set_ids[0]
print(f"Using record set @id='{active_record_set_id}' for further analysis.")
print(f"First five columns: {dataframes[active_record_set_id].columns[:5].tolist()}")
dataframes[active_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping data by key attributes to prepare for further analysis.

All field/column references use their `@id` as shown above.

In [ ]:
# Pick a numeric field for analysis. Replace below with the actual @id after inspecting earlier output.

# You may use the overview above, or set manually like:
numeric_field = None

# Find a likely numeric field from the RecordSet fields
for field in metadata.record_set[0].field:
    if field.data_type in ['Float', 'Integer', 'Number']:
        numeric_field = field.id
        break

if numeric_field is None or numeric_field not in dataframes[active_record_set_id].columns:
    print("No numeric field found for demonstration.")
else:
    print(f"Using numeric field: {numeric_field}")
    threshold = dataframes[active_record_set_id][numeric_field].mean()
    filtered_df = dataframes[active_record_set_id][dataframes[active_record_set_id][numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try a group-by using a categorical field if present
    # Attempt to find a non-numeric field
    group_field = None
    for field in metadata.record_set[0].field:
        if field.data_type.lower() not in ['float', 'integer', 'number'] and field.id in filtered_df.columns:
            group_field = field.id
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No suitable group field found for aggregation.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All fields are referenced using their `@id`.

Examples below show a histogram and a box plot if numeric columns are present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in dataframes[active_record_set_id].columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[active_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if possible
    if group_field and group_field in dataframes[active_record_set_id].columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[active_record_set_id])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The `mlcroissant` library enables structured exploration and extraction of FAIR datasets referenced by Croissant schema.
- We demonstrated loading metadata, record set enumeration, field inspection (all via `@id`), and performed basic exploratory data analysis and visualization.
- This workflow can be adapted for further domain-specific analysis.

For further details on this dataset, refer to its [schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) or the original data documentation.